# Evaluate a trained VLG-CBM

Logic lives in `vlgcbm_analysis.py`. Use the **Python (vlgcbm)** kernel.

Note on splits: birds525 ships `train` / `val` / `test`, and `val` and `test` are
different images. Both repos map `birds525_val` to the `val` folder, so `SPLIT="val"`
below is what every reported number uses; the `test` folder is unused.

In [ ]:
import os
os.environ.setdefault("DATASET_FOLDER", "/workspace/VLG-CBM/datasets")
os.environ.setdefault("VLGCBM_BIOCLIP_CKPT", "/workspace/models/bioclip/open_clip_pytorch_model.bin")

import torch
import vlgcbm_analysis as va

print("device:", "cuda" if torch.cuda.is_available() else "cpu")
print("models:", ", ".join(va.list_models("birds525")))

In [ ]:
MODEL   = "bioclip"      # <-- swap me
DATASET = "birds525"
SPLIT   = "val"

run = va.load_run(va.run_dir(MODEL, DATASET))
res = va.evaluate(run, split=SPLIT)          # cached to <run>/eval_val.pt
print(run)
print("accuracy: {:.2f}%".format(res.accuracy * 100))

In [ ]:
best, worst = va.best_worst_classes(run, res, k=5)
print("best :", best)
print("worst:", worst)

In [ ]:
va.sankey_static(run, [best[0], worst[0]],
                 weight_cutoff=0.05, max_per_class=12,
                 save_path="figures/sankey.png")

## 1. Pick a class

`correct/support` is what to sort on. birds525's val split has 5 images per class, so
accuracy takes only 6 values and large groups of classes tie at 100% and 0%.

In [ ]:
rows = va.class_table(run, res, ascending=False, n=15)   # strongest

In [ ]:
rows = va.class_table(run, res, ascending=True, n=15)    # weakest

## 2. Single example

`class_indices` gives that class's eval-set indices; `only="wrong"` restricts to errors.

In [ ]:
CLASS = "wood duck"        # <-- from the tables above

idx_all   = va.class_indices(res, CLASS)
idx_wrong = va.class_indices(res, CLASS, only="wrong")
print(f"{CLASS}: {len(idx_all)} images, {len(idx_wrong)} wrong")
print("indices:", list(map(int, idx_all)))

In [ ]:
IDX = int(idx_all[0])      # <-- or pick from the list above
va.explain_example(run, idx=IDX, split=SPLIT)

## 3. Same example, with boxes

Boxes are Grounding DINO's, at the threshold the run was trained with. A top concept
marked `(no box)` was never detected in this image -- it is acting as a class prior
rather than as visual evidence.

In [ ]:
va.explain_with_boxes(run, idx=IDX, split=SPLIT, top_k=8)

## 4. Sankey: concept -> class

Final-layer weights as concept -> class flows. Contrasting a strong class against a weak
one is usually the informative view.

In [ ]:
va.sankey_static(run, [CLASS, rows[0]["class"]],
                 weight_cutoff=0.05, max_per_class=12,
                 save_path="figures/sankey.png")

## 5. Model comparison grid

In [ ]:
outs = va.evaluate_many(va.find_runs(), split=SPLIT, keep_concept_acts=True)
for o in outs:
    print(o)

In [ ]:
va.story_figure(outs, split=SPLIT, top_concepts=2,
                flag_mode="sufficiency",
                save_path="figures/qualitative_comparison.png")